# 3.2 · 缺失值处理 / Handling Missing Values

> **课程定位 / Where this fits**
> 第 2 课，**Part 3 · EDA 与数据预处理**。
> Lesson 2, **Part 3 · EDA & Preprocessing**.
>
> 3.1 的 EDA 发现了缺失值，这一课系统讲怎么处理。缺失值处理是实战里最常见、最容易踩坑的一步——填得不对会**悄悄引入偏差**，还可能**泄漏**。
> EDA (3.1) found missing values; this lesson handles them systematically. It's one of the most common and trap-laden steps in practice — done wrong, it **silently injects bias** and can even **leak**.
>
> 💼 **实战/面试视角**：面试爱问"缺失值怎么处理 / 三种缺失机制 / 为什么均值填补不好"。
> 💼 **Practical/interview angle:** interviewers love "how do you handle missing data / the three missingness mechanisms / why is mean imputation bad?"

> 📐 **符号约定 / Notation**
> - MCAR / MAR / MNAR —— 三种缺失机制（见第 2 节）/ the three missingness mechanisms
> - 填补 imputation —— 用某个值替换缺失 / replacing a missing value with an estimate

> 💡 **面试相关 / Interview-relevant**
> - "MCAR / MAR / MNAR 三种缺失机制的区别"（出镜率 ★★★★★）
> - "为什么均值填补不好"（★★★★★，缩方差 + 扭曲相关）
> - "KNN / MICE 填补的原理与取舍"（★★★★）
> - "缺失本身是不是信息（缺失指示符）"（★★★★）
> - "填补会不会泄漏 / 在哪一步做"（★★★★★，只 fit 训练集）

---

## 学习目标 / Learning Objectives

1. 区分 **MCAR / MAR / MNAR** 三种缺失机制及其后果。
   Distinguish MCAR / MAR / MNAR and their consequences.
2. 知道何时**删除**、何时**填补**。
   Know when to **delete** vs **impute**.
3. 看清**均值填补的两宗罪**（缩方差、扭曲相关）。
   See the **two sins of mean imputation** (shrinks variance, distorts correlation).
4. 用**分组填补 / KNN / MICE** 做更聪明的填补。
   Impute smarter with **grouped / KNN / MICE**.
5. 把"缺失"本身当作特征（**缺失指示符**），并**防泄漏**（只 fit 训练集）。
   Treat missingness as a feature (**indicator**) and **prevent leakage** (fit on train only).

## 目录 / TOC
1. [先建直觉 + 三种缺失机制 ⭐](#1)
2. [🚢 数据：缺失诊断](#2)
3. [删除：什么时候能删](#3)
4. [均值填补的两宗罪 ⭐](#4)
5. [分组填补 / KNN / MICE ⭐](#5)
6. [缺失指示符：缺失也是信息 ⭐](#6)
7. [防泄漏：只 fit 训练集 ⭐](#7)
8. [实战对比 + 小结](#8)


<a id="1"></a>
## 1. 先建直觉 + 三种缺失机制 ⭐ / Intuition & the Three Mechanisms

填补缺失值，本质是在**猜一个看不到的真值**。猜得好不好，关键取决于"**为什么会缺**"——这就是三种缺失机制（统计学家 Rubin 的分类，面试必考）：
Imputation is essentially **guessing an unseen true value**. How well you can guess depends on **why it's missing** — the three missingness mechanisms (Rubin's classification, a must-know):

- **MCAR（完全随机缺失）**：缺不缺**纯随机**，和任何东西都无关（如仪器随机故障）。最幸运——删掉缺失行也不会引入偏差。
  **MCAR (Missing Completely At Random):** missingness is **pure chance**, unrelated to anything (e.g. random sensor glitch). The luckiest case — deleting missing rows introduces no bias.
- **MAR（随机缺失）**：缺不缺**取决于其他观测到的变量**（如老年人更不愿填收入，缺失依赖 age）。**可以用其他列辅助填补**来纠偏。
  **MAR (Missing At Random):** missingness **depends on other observed variables** (e.g. older people skip income → missingness depends on age). You **can use other columns to impute** and correct the bias.
- **MNAR（非随机缺失）**：缺不缺**取决于缺失值本身**（如高收入者更不愿填收入）。最棘手——光靠数据填不准，需要业务知识或建模假设。
  **MNAR (Missing Not At Random):** missingness **depends on the missing value itself** (e.g. high earners hide income). The hardest — data alone can't fix it; needs domain knowledge.

下面用**真值已知**的合成数据演示三种机制：看"观测到的均值"如何偏离真实均值。
Below we use synthetic data with **known truth** to show each mechanism: how the "observed mean" drifts from the true mean.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 造一份真值已知的数据: 收入与年龄正相关 / known-truth data: income correlates with age
n = 2000
age = rng.uniform(20, 80, n)
income = 20 + 0.8*age + rng.normal(0, 8, n)

def make_missing(mechanism):
    inc = income.copy()
    if mechanism == "MCAR":
        mask = rng.random(n) < 0.3                          # 纯随机缺失 / pure random
    elif mechanism == "MAR":
        mask = rng.random(n) < (age - 20)/60 * 0.6          # 年龄越大越易缺(依赖 age) / depends on age
    else:  # MNAR
        mask = rng.random(n) < (income - income.min())/(income.max()-income.min()) * 0.6  # 收入越高越易缺(依赖自身)
    inc[mask] = np.nan
    return inc

data = {m: make_missing(m) for m in ["MCAR","MAR","MNAR"]}
print(f"真实 income 均值 true mean = {income.mean():.2f}")
for m, inc in data.items():
    # nanmean: 忽略 NaN 求均值; 看观测均值偏离真值多少 / observed mean vs truth
    print(f"{m}: 缺失 missing {np.isnan(inc).mean():.0%}, 观测均值 observed mean = {np.nanmean(inc):.2f}")
print("→ MCAR 观测均值≈真值; MAR/MNAR 明显偏低(缺的恰是高收入/老年人) → 直接删/均值填补会有偏")


<a id="2"></a>
## 2. 数据：缺失诊断 / Diagnosing Missingness

回到 **Titanic**。诊断缺失的第一步是看**缺失率**，第二步是判断"缺失是否依赖其他变量"——如果是，就是 MAR 的线索，可以用那个变量辅助填补。
Back to **Titanic**. Step one is the **missing rate**; step two is checking whether missingness depends on other variables — if so, that's a MAR clue and you can use that variable to impute.


In [ ]:
df = sns.load_dataset("titanic")
print("各列缺失率 missing rate (%):")
print((df.isna().mean()*100).round(1).sort_values(ascending=False).head())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# 缺失地图: 每个单元格缺失=1(亮色) / missingness map: bright = missing
sns.heatmap(df[["age","deck","embarked","embark_town"]].isna(), cbar=False,
            yticklabels=False, cmap="viridis", ax=axes[0])
axes[0].set_title("缺失地图 missingness map (亮=缺失 bright=missing)")

# age 的缺失率是否随 pclass 变化? 变化→不是MCAR, 是MAR线索 / does age-missing depend on pclass?
miss_by_class = df.assign(age_missing=df.age.isna()).groupby("pclass")["age_missing"].mean()
miss_by_class.plot(kind="bar", ax=axes[1])
axes[1].set_title("age 缺失率 by pclass — 三等舱更高 → MAR 线索 MAR clue")
axes[1].set_ylabel("missing rate")
plt.tight_layout(); plt.show()
print("\nage 缺失率随舱等变化 → 不是 MCAR, 至少是 MAR (可用 pclass 辅助填补)")
print("age-missingness varies by class → not MCAR, at least MAR (use pclass to help impute).")


<a id="3"></a>
## 3. 删除：什么时候能删 / Deletion: When It's OK

最简单的处理是**删除**，但要小心：
The simplest fix is **deletion**, but be careful:
- **listwise 删除（删整行）**：只要任一列缺失就删掉整行。MCAR 下无偏，但**很浪费数据**——Titanic 全删后只剩 ~20% 行。
  **Listwise deletion:** drop a row if any column is missing. Unbiased under MCAR but **wastes data** — Titanic keeps only ~20% of rows.
- **删列**：某列缺失太多（如 deck 缺 77%），填补也没意义，直接删列更干净。
  **Drop a column:** if a column is mostly missing (deck is 77% missing), imputing is pointless — drop the column.
- **删行（针对单列）**：只删某关键列缺失的行，损失小。
  **Drop rows for one column:** drop only rows missing a key column — smaller loss.


In [ ]:
print(f"Titanic 原始 original: {len(df)} 行 rows")
print(f"listwise 删除所有缺失 drop any-missing: {len(df.dropna())} 行 (只剩 {len(df.dropna())/len(df):.0%}!) — 太狠 too aggressive")
print(f"只删 age 缺失行 drop age-missing only: {len(df.dropna(subset=['age']))} 行")
print("合理方案 sensible: 删 deck 列(77%缺) + 对 age 用填补(下文) → 保住大部分数据")


<a id="4"></a>
## 4. 均值填补的两宗罪 ⭐ / The Two Sins of Mean Imputation

"缺了就填均值"是最常见、也最该被警惕的做法。哪怕在最干净的 MCAR 下，它都有两宗罪（面试高频）：
"Just fill with the mean" is the most common — and most cautioned-against — approach. Even under the cleanest MCAR, it commits two sins (frequently asked):


In [ ]:
from sklearn.impute import SimpleImputer

# 用 MCAR 数据(机制最干净, 排除机制偏差, 单看方法本身的问题) / use clean MCAR to isolate the method's flaw
inc_mcar = data["MCAR"].reshape(-1, 1)
observed = inc_mcar[~np.isnan(inc_mcar)]                                  # 非缺失的观测值
mean_filled = SimpleImputer(strategy="mean").fit_transform(inc_mcar).ravel()  # 均值填补

print("=== 罪一: 缩小方差 sin #1: shrinks variance ===")
print(f"真实 true std         = {income.std():.2f}")
print(f"观测(非缺) std         = {observed.std():.2f}")
print(f"均值填补后 std         = {mean_filled.std():.2f}  ← 被人为压低 artificially shrunk!")
print("原因: 填进去的全是同一个值(均值), 方差当然变小 → 低估了不确定性")

print("\n=== 罪二: 扭曲相关 sin #2: distorts correlation ===")
df_corr = pd.DataFrame({"age": age, "income": data["MCAR"]})
true_corr = np.corrcoef(age, income)[0,1]
# 用均值填补 income 后, 填进去的值与 age 无关 → 把真实相关稀释了
filled_corr = np.corrcoef(age, SimpleImputer(strategy="mean").fit_transform(df_corr[["income"]]).ravel())[0,1]
print(f"真实 age-income 相关 true corr = {true_corr:.3f}")
print(f"均值填补后相关 after mean-impute = {filled_corr:.3f}  ← 被稀释 diluted (填的值与 age 无关)")


<a id="5"></a>
## 5. 分组填补 / KNN / MICE ⭐ / Smarter Imputation

均值填补忽略了"其他列的信息"。更聪明的填补会**利用相关变量**（这正是纠正 MAR 偏差的关键）：
Mean imputation ignores other columns. Smarter methods **use correlated variables** (the key to correcting MAR bias):

- **分组填补 / grouped**：按相关的类别列分组，用组内中位数填（如各舱等用各自的年龄中位数）。
  Group by a related categorical column and fill with the group median (e.g. each class's own age median).
- **KNN 填补**：用其他特征找最相似的 K 个邻居，取它们的值填。注意**对尺度敏感，要先标准化**（同 5.3 KNN）。
  **KNN:** find the K most similar rows by other features and use their values. **Scale-sensitive — standardize first** (like KNN, 5.3).
- **MICE（多重插补 / IterativeImputer）**：把每个缺失列当作目标，用其他列**回归预测**它，迭代多轮。利用全部列间关系，MAR 下接近无偏，还保留变异性。
  **MICE (IterativeImputer):** treat each missing column as a target and **regress** it on the others, iterating. Uses all inter-column relationships, near-unbiased under MAR, and keeps variability.


In [ ]:
# 分组填补: 各舱等的年龄中位数差异明显 → 分组填补更准 / grouped imputation
df_g = df.copy()
df_g["age_global"] = df_g["age"].fillna(df_g["age"].median())   # 全局中位数填补
# groupby+transform: 在每个 pclass 组内用组内中位数填该组的缺失 / fill within each class
df_g["age_grouped"] = df_g.groupby("pclass")["age"].transform(lambda s: s.fillna(s.median()))
print("各舱等年龄中位数 median age per class (差异明显):")
print(df.groupby("pclass")["age"].median())
print(f"\n全局填补值 global: {df.age.median():.0f} (所有缺失都填这个);  分组填补: 一等~37 / 三等~24 (尊重 MAR 结构)")


In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa  必须先 import 这个开关, 才能 import IterativeImputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler

num_cols = ["age","fare","pclass","sibsp","parch"]
X = df[num_cols].copy()

# KNN: 先标准化(KNN 对尺度敏感), 再用 5 个邻居填补 / standardize then KNN-impute
Xs = StandardScaler().fit_transform(X)
X_knn = KNNImputer(n_neighbors=5).fit_transform(Xs)
age_knn = X_knn[df["age"].isna().values, 0]    # 取被填补的 age(标准化后的值)
print(f"KNN 填补的 age (标准化后)范围: [{age_knn.min():.2f}, {age_knn.max():.2f}] — 有变化(非常量), 不缩方差")

# MICE: 用其他列回归预测 age 等缺失列, 迭代 10 轮 / regress-based iterative imputation
X_mice = IterativeImputer(max_iter=10, random_state=0).fit_transform(X)
age_mice = X_mice[df["age"].isna().values, 0]  # 取被填补的 age(原始尺度)
print(f"MICE 填补的 age: 均值 {age_mice.mean():.1f}, std {age_mice.std():.1f} — 保留了变异性(对比均值填补 std=0)")
print("\nKNN 优点: 填补值有变化→不缩方差, 多特征找邻居→更准; 缺点: O(n²)慢, 高维失效")
print("MICE 优点: 利用全部列间关系, MAR 下接近无偏; 缺点: 慢, 默认假设线性")


<a id="6"></a>
## 6. 缺失指示符：缺失也是信息 ⭐ / Missingness as a Feature

一个常被忽视但很实用的技巧：**"某个值缺失"这件事本身可能就携带信息**。比如"没填年龄"的乘客可能是某类特殊人群，生还率不同。做法：填补的同时，**额外加一列 0/1 的"缺失指示符"**，让模型自己决定要不要用。
An often-overlooked but practical trick: **the fact that a value is missing can itself be informative.** E.g. passengers with no recorded age might be a distinct group with a different survival rate. The method: while imputing, **add a 0/1 "missing indicator" column** and let the model decide whether to use it.


In [ ]:
from sklearn.impute import SimpleImputer

# add_indicator=True: 填补的同时, 自动追加"是否缺失"的 0/1 指示列 / impute + add missingness flags
imp = SimpleImputer(strategy="median", add_indicator=True)
X_ind = imp.fit_transform(df[["age","fare"]])
print(f"原 2 列 → 填补后 {X_ind.shape[1]} 列 ({X_ind.shape[1]-2} 个新增的缺失指示符)")

# 验证: "age 缺失"本身是否和生还有关(决定加指示符是否有价值) / is missingness informative?
df_chk = df.assign(age_missing=df.age.isna())
print(f"\nage 缺失者生还率 missing-age survival: {df_chk[df_chk.age_missing]['survived'].mean():.2%}")
print(f"age 已知者生还率 known-age survival:   {df_chk[~df_chk.age_missing]['survived'].mean():.2%}")
print("→ 两者有差异! 'age 缺失'本身携带信息 → 加缺失指示符有价值")
print("→ Different! missingness carries signal → the indicator is worth adding.")


<a id="7"></a>
## 7. 防泄漏：只 fit 训练集 ⭐ / Prevent Leakage: Fit on Train Only

**这是缺失值处理最重要的实战纪律**：填补用的统计量（均值/中位数/KNN 邻居/回归模型）**只能从训练集学**，再用同一个值去 transform 测试集。如果在全数据上算中位数，就把测试集的信息泄漏进了训练——评估会虚高，上线就翻车（接 3.9 / 3.12）。
**This is the single most important discipline here:** the statistics used to impute (mean/median/KNN neighbors/regression model) must be **learned from the training set only**, then applied to the test set. Computing the median on all data leaks test information into training — your evaluation looks too good and production fails (see 3.9 / 3.12).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X = df[["age","fare"]]; y = df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

# ❌ 错误: 在全数据(含 test)上 fit 填补器 / WRONG: fit on all data including test
wrong = SimpleImputer(strategy="median").fit(X)

# ✅ 正确: fit 只看 train, test 用 train 学到的中位数来 transform / RIGHT: fit on train only
imp = SimpleImputer(strategy="median").fit(X_tr)
X_tr_imp = imp.transform(X_tr)
X_te_imp = imp.transform(X_te)            # 注意: 用的是 train 的中位数, 不是 test 自己的

print(f"全数据 age 中位数 (错误用) all-data: {wrong.statistics_[0]:.1f}")
print(f"仅 train age 中位数 (正确)  train-only: {imp.statistics_[0]:.1f}")
print("差异虽小, 但原则必须遵守 — test 的任何统计量都不能渗入训练")
print("💡 用 Pipeline (3.12) 把填补和模型打包, 自动保证只 fit 训练折, 永不手滑")


<a id="8"></a>
## 8. 实战对比 + 小结 / Showdown & Summary

在 **MAR** 数据上（缺失依赖 age），对比各方法恢复真实均值/方差的能力——这是最能体现方法差异的场景。
On **MAR** data (missingness depends on age), compare how well each method recovers the true mean/variance — the scenario that best exposes the differences.


In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

df_mar = pd.DataFrame({"age": age, "income": data["MAR"]})   # 缺失依赖 age 的 MAR 数据
true_mean, true_std = income.mean(), income.std()

results = {}
results["均值 mean"] = SimpleImputer(strategy="mean").fit_transform(df_mar[["income"]]).ravel()
Xs = StandardScaler().fit_transform(df_mar)                  # KNN 用 age 找邻居(需标准化)
results["KNN(用age)"] = KNNImputer(n_neighbors=10).fit_transform(Xs)[:,1]*df_mar.income.std()+np.nanmean(df_mar.income)
results["MICE(用age)"] = IterativeImputer(random_state=0).fit_transform(df_mar)[:,1]  # 用 age 回归

print(f"真实 truth: mean={true_mean:.2f}, std={true_std:.2f}\n")
print(f"{'方法 method':<16} {'mean':>8} {'std':>8}")
print(f"{'观测(非缺)':<16} {np.nanmean(df_mar.income):>8.2f} {np.nanstd(df_mar.income):>8.2f}  ← 有偏(缺的是老/高收入)")
for k, v in results.items():
    print(f"{k:<16} {v.mean():>8.2f} {v.std():>8.2f}")
print("\nMICE/KNN 利用 age 纠正了 MAR 偏差并保留方差; 均值填补缩小了 std 且没纠偏")


```
三机制: MCAR(纯随机, 删除无偏) / MAR(依赖其他观测列, 可辅助填补) / MNAR(依赖缺失值本身, 最难)
删除: 删列(高缺失) 或 删行(单关键列); listwise 删全部很浪费
均值填补两宗罪: ① 缩小方差(填的全是同一值) ② 扭曲相关(填值与其他列无关)
更聪明: 分组中位数 / KNN(需标准化) / MICE(回归迭代, 利用列间关系, 保留变异)
缺失指示符: 缺失本身可能是信息 → 加 0/1 列让模型用
防泄漏 ⭐: 填补统计量只从 train 学; 用 Pipeline(3.12) 保证
```

### 💡 面试速查 / Interview cheat-sheet
1. **MCAR/MAR/MNAR**：纯随机 / 依赖其他列 / 依赖自身；MAR 可用其他列纠偏。
   Pure random / depends on others / depends on itself; MAR is fixable using other columns.
2. **均值填补缩方差 + 扭曲相关**；连最干净的 MCAR 下都有问题。
   Mean imputation shrinks variance and distorts correlation, even under clean MCAR.
3. **KNN 要先标准化**；**MICE 用回归迭代**，保留变异、近无偏。
   KNN needs standardizing; MICE regresses iteratively, keeping variability.
4. **缺失指示符**：缺失本身可能携带信号。
   The missingness indicator: missingness can carry signal.
5. **填补只 fit 训练集**，否则泄漏；Pipeline 自动保证。
   Fit imputation on train only or you leak; Pipelines enforce it.

### 下一节 / Next
**3.3 异常值检测**——缺失之外的另一类脏数据：极端值。怎么发现(IQR/Z-score/孤立森林)、怎么处理(截断/变换/标记)。
**3.3 Outliers** — the other kind of dirty data: extreme values. How to detect (IQR/Z-score/Isolation Forest) and handle (clip/transform/flag).
